# PETR4 × Brent × OVX: previsão da volatilidade futura de 5 pregões

Reprodução executada em 24/09/2026 do protocolo do notebook `trab07_V4_HV30_OVX.ipynb`. O alvo original chama-se `PETR4_Vol_Observada_5d` (equivalente conceitual a `PETR4_Vol_Futura_5d`). Corrigimos o nome incompatível no bloco OVX e usamos `PETR4_Alvo_Fim < Date` no treino, em vez de remover cinco linhas por posição.

**Alteração de fonte:** Yahoo/yfinance respondeu HTTP 429. PETR4 veio da [brapi](https://brapi.dev/docs/acoes), via `/api/quote/PETR4?range=max&interval=1d`; Brent e OVX vieram das séries FRED `DCOILBRENTEU` e `OVXCLS`. A brapi fornece `adjustedClose` e OHLC/volume; as máximas, mínimas e abertura foram multiplicadas por `adjustedClose/close`, reproduzindo a convenção `auto_adjust=True`. A série da brapi pode diferir da série histórica atual do Yahoo. Cotações e ajustes de provedores não constituem reconstrução ponto no tempo.

O teste 2024–2025 permaneceu reservado quanto a escolha de atributos e modelo. Não há HV30 do OVX.

In [1]:
import requests, pandas as pd, numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

r=requests.get('https://brapi.dev/api/quote/PETR4?range=max&interval=1d',timeout=60)
r.raise_for_status()
raw=pd.DataFrame(r.json()['results'][0]['historicalDataPrice'])
raw['Date']=pd.to_datetime(raw.date,unit='s',utc=True).dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None).dt.normalize()
raw=raw[(raw.Date>='2016-01-01')&(raw.Date<='2025-12-31')].set_index('Date').sort_index()
def fred(series):
    url=f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}&cosd=2016-01-01&coed=2025-12-31'
    q=pd.read_csv(url,parse_dates=['observation_date']).set_index('observation_date')
    q.index.name='Date'
    return q
fred_brent=fred('DCOILBRENTEU')
fred_ovx=fred('OVXCLS')
print('PETR4:',len(raw),'Brent:',len(fred_brent),'OVX:',len(fred_ovx))
print('PETR4:',raw.index.min().date(),raw.index.max().date(),'volume zero:',(raw.volume==0).sum())
print('Brent ausentes:',fred_brent.DCOILBRENTEU.isna().sum(),'OVX ausentes:',fred_ovx.OVXCLS.isna().sum())

PETR4: 2483 Brent: 2608 OVX: 2608
PETR4: 2016-01-04 2025-12-30 volume zero: 0
Brent ausentes: 70 OVX ausentes: 92


## Engenharia de atributos e integração temporal

Desvio-padrão populacional (`ddof=0`), alvo nos próximos cinco pregões, e defasagem de **uma observação válida** em cada série externa antes do `merge_asof` por calendário da PETR4.

In [2]:
factor = raw.adjustedClose / raw.close
p = pd.DataFrame(index=raw.index)
for c in ['open', 'high', 'low', 'close']:
    p['PETR4_'+c.title()] = raw[c] * factor
p['PETR4_Volume'] = raw.volume
p = p[p.PETR4_Volume > 0]
p['PETR4_Retorno'] = np.log(p.PETR4_Close / p.PETR4_Close.shift())
p['PETR4_Amplitude'] = np.log(p.PETR4_High / p.PETR4_Low)
p['PETR4_Var_Volume'] = np.log(p.PETR4_Volume / p.PETR4_Volume.shift())
p['PETR4_HV30'] = p.PETR4_Retorno.rolling(30).std(ddof=0)
future = pd.concat([p.PETR4_Retorno.shift(-i) for i in range(1,6)], axis=1)
p['PETR4_Vol_Observada_5d'] = future.std(axis=1,ddof=0,skipna=False)*100
p['PETR4_Alvo_Fim'] = pd.Series(p.index,index=p.index).shift(-5)

def external(name, value_col, ret_col, hv=False):
    d = {'DCOILBRENTEU':fred_brent, 'OVXCLS':fred_ovx}[name].copy()
    d.index.name='Date'
    d[value_col] = pd.to_numeric(d[name],errors='coerce')
    d = d.dropna(subset=[value_col])
    d[ret_col] = np.log(d[value_col] / d[value_col].shift())
    cols=[ret_col]
    if hv:
        d[name+'_HV30'] = d[ret_col].rolling(30).std(ddof=0)
        cols += [name+'_HV30']
    else:
        cols=[value_col,ret_col]
    d[name+'_Origem'] = d.index
    d[cols+[name+'_Origem']] = d[cols+[name+'_Origem']].shift(1)
    return d[cols+[name+'_Origem']].reset_index().sort_values('Date')

b=external('DCOILBRENTEU','BRENT_Spot','BRENT_Retorno',True)
o=external('OVXCLS','OVX_Nivel','OVX_Retorno')
d=p.reset_index().sort_values('Date')
d=pd.merge_asof(d,b,on='Date',direction='backward')
d=pd.merge_asof(d,o,on='Date',direction='backward')
pet=['PETR4_Retorno','PETR4_Amplitude','PETR4_Var_Volume','PETR4_HV30']
brent=pet+['BRENT_Retorno','DCOILBRENTEU_HV30']
ovx=pet+['OVX_Nivel','OVX_Retorno']
models={'PETR4':pet,'PETR4 + Brent':brent,'PETR4 + OVX':ovx}
target='PETR4_Vol_Observada_5d'

def walk(frame, modelset=models):
    frame=frame.dropna(subset=[target,'PETR4_HV30']).copy().sort_values('Date')
    out=[]
    for row in frame.itertuples(index=False):
        t=row.Date
        if t<pd.Timestamp('2017-01-01') or t>pd.Timestamp('2025-12-31'):continue
        train=frame[frame.PETR4_Alvo_Fim < t]
        record={'Date':t,'Real':getattr(row,target),'Baseline HV30':getattr(row,'PETR4_HV30')*100,
                'N_Treino':len(train),'Ultimo_Alvo_Conhecido':train.PETR4_Alvo_Fim.max()}
        current=frame.loc[frame.Date==t]
        for name,features in modelset.items():
            tr=train.dropna(subset=features)
            if current[features].isna().any(axis=None):record[name]=np.nan;continue
            reg=make_pipeline(StandardScaler(),LinearRegression())
            reg.fit(tr[features],tr[target])
            record[name]=reg.predict(current[features])[0]
        out.append(record)
    return pd.DataFrame(out)

def metrics(x, cols):
    return pd.DataFrame([{'Modelo':k,'N':len(x),'RMSE':mean_squared_error(x.Real,x[k])**.5,
         'MAE':mean_absolute_error(x.Real,x[k]),'R2':r2_score(x.Real,x[k]),
         'Bias':(x[k]-x.Real).mean()} for k in cols])


print('Base:',len(d),'Brent no futuro:',(d.DCOILBRENTEU_Origem>=d.Date).sum(),'OVX no futuro:',(d.OVXCLS_Origem>=d.Date).sum())
print('Alvos completos:',d[target].notna().sum(),'amostra comum:',len(d.dropna(subset=brent+ovx+[target])))

Base: 2483 Brent no futuro: 0 OVX no futuro: 0
Alvos completos: 2478 amostra comum: 2448


## Reprodução de Brent e comparação comum

Treino expansivo com `StandardScaler` ajustado de novo somente no treino de cada data. São elegíveis apenas linhas cujo último dia do alvo precede estritamente a data da previsão. **Bias = previsto − observado**, em pontos percentuais.

In [3]:
cols=['Baseline HV30']+list(models)
original=d.dropna(subset=brent+[target]).copy()
res_original=walk(original,{'PETR4':pet,'PETR4 + Brent':brent})
common=d.dropna(subset=brent+ovx+[target]).copy()
res=walk(common)
assert (res.Ultimo_Alvo_Conhecido < res.Date).all()
assert (res_original.Ultimo_Alvo_Conhecido < res_original.Date).all()
assert (common.DCOILBRENTEU_Origem < common.Date).all()
assert (common.OVXCLS_Origem < common.Date).all()
print('Reprodução Brent (N de previsões):',len(res_original))
display(metrics(res_original[res_original.Date.dt.year<=2023],cols[:-1]).round(4))
display(metrics(res_original[res_original.Date.dt.year>=2024],cols[:-1]).round(4))
referencia=pd.DataFrame({'Modelo':cols[:-1], 'Histórico publicado':[1.7674,1.4393,1.4666], 'Teste publicado':[.8029,.7833,.7831]})
referencia['Histórico reproduzido']=metrics(res_original[res_original.Date.dt.year<=2023],cols[:-1]).RMSE
referencia['Teste reproduzido']=metrics(res_original[res_original.Date.dt.year>=2024],cols[:-1]).RMSE
display(referencia.round(4))
print('Datas comuns PETR4/Brent/OVX:',len(res),'2017–2023:',sum(res.Date.dt.year<=2023),'2024–2025:',sum(res.Date.dt.year>=2024))

Reprodução Brent (N de previsões): 2229
Datas comuns PETR4/Brent/OVX: 2229 2017–2023: 1733 2024–2025: 496


,Modelo,N,RMSE,MAE,R2,Bias
0,Baseline HV30,1733,1.7684,1.0668,-0.2716,0.4199
1,PETR4,1733,1.4322,0.8526,0.1659,0.1195
2,PETR4 + Brent,1733,1.4562,0.8686,0.1378,0.0920


,Modelo,N,RMSE,MAE,R2,Bias
0,Baseline HV30,496,0.8394,0.6516,-0.5342,0.2983
1,PETR4,496,0.8093,0.6559,-0.4262,0.4022
2,PETR4 + Brent,496,0.8094,0.6556,-0.4263,0.4065


,Modelo,Histórico publicado,Teste publicado,Histórico reproduzido,Teste reproduzido
0,Baseline HV30,1.7674,0.8029,1.7684,0.8394
1,PETR4,1.4393,0.7833,1.4322,0.8093
2,PETR4 + Brent,1.4666,0.7831,1.4562,0.8094


In [4]:
historico=metrics(res[res.Date.dt.year<=2023],cols)
teste=metrics(res[res.Date.dt.year>=2024],cols)
print('Histórico 2017–2023 (amostra comum)');display(historico.round(4))
print('Teste reservado 2024–2025 (amostra comum)');display(teste.round(4))
def reducoes(tab):
    m=tab.set_index('Modelo').RMSE
    return pd.Series({'Brent contra PETR4':100*(m['PETR4']-m['PETR4 + Brent'])/m['PETR4'],
      'OVX contra PETR4':100*(m['PETR4']-m['PETR4 + OVX'])/m['PETR4'],
      'OVX contra Brent':100*(m['PETR4 + Brent']-m['PETR4 + OVX'])/m['PETR4 + Brent']})
print('Reduções percentuais de RMSE (positivo = melhora)')
display(pd.DataFrame({'2017–2023':reducoes(historico),'2024–2025':reducoes(teste)}).round(2))
anual=pd.DataFrame([{'Ano':int(y),'Modelo':k,'RMSE':mean_squared_error(g.Real,g[k])**.5}
   for y,g in res.groupby(res.Date.dt.year) for k in cols]).pivot(index='Ano',columns='Modelo',values='RMSE')
print('RMSE por ano, mesmas datas');display(anual.round(4))

Histórico 2017–2023 (amostra comum)
Teste reservado 2024–2025 (amostra comum)
Reduções percentuais de RMSE (positivo = melhora)
RMSE por ano, mesmas datas


,Modelo,N,RMSE,MAE,R2,Bias
0,Baseline HV30,1733,1.7684,1.0668,-0.2716,0.4199
1,PETR4,1733,1.4322,0.8526,0.1659,0.1195
2,PETR4 + Brent,1733,1.4562,0.8686,0.1378,0.0920
3,PETR4 + OVX,1733,1.4919,0.8855,0.0949,0.0767


,Modelo,N,RMSE,MAE,R2,Bias
0,Baseline HV30,496,0.8394,0.6516,-0.5342,0.2983
1,PETR4,496,0.8093,0.6559,-0.4262,0.4022
2,PETR4 + Brent,496,0.8094,0.6556,-0.4263,0.4065
3,PETR4 + OVX,496,0.8147,0.6623,-0.4451,0.4106


,2017–2023,2024–2025
Brent contra PETR4,-1.67,-0.00
OVX contra PETR4,-4.17,-0.66
OVX contra Brent,-2.46,-0.66


Modelo,Baseline HV30,PETR4,PETR4 + Brent,PETR4 + OVX
Ano,,,,
2017,1.3874,1.2088,1.1917,1.1058
2018,1.8987,1.5276,1.5627,1.7098
2019,0.8946,0.7994,0.8315,0.8662
2020,3.1372,2.3970,2.4673,2.5432
2021,2.0044,1.5734,1.5766,1.5820
2022,0.9861,1.0232,1.0257,1.0255
2023,0.8481,0.8064,0.8105,0.8101
2024,0.8976,0.8692,0.8713,0.8759
2025,0.7753,0.7430,0.7406,0.7467


## Interpretação e limitações

Os números reproduzidos divergem materialmente do teste original (especialmente em 2024); as séries ajustadas da brapi e do Yahoo e o calendário disponível não são idênticos. A amostra comum elimina diferenças de calendário **entre os modelos desta reprodução**, mas não recupera a comparabilidade exata com o antigo Yahoo. O experimento original relatou 2.481 pregões válidos de PETR4 e 2.222 previsões; esta coleta alternativa tem 2.483 pregões e 2.229 previsões. FRED pode revisar séries passadas; dados de fechamento do OVX ainda podem ter horário de publicação posterior à observação, razão da defasagem conservadora.

Os alvos de cinco pregões sucessivos se sobrepõem; RMSE anual não deve ser interpretado como amostras independentes. A comparação é observacional e limitada a regressão linear, atributos especificados e uma versão atualizada dos provedores. A conclusão sobre OVX refere-se somente à amostra comum da presente execução.